# PokéPredictors: Pokémon Battle Prediction

FDS 2026 Kaggle challenge, team PokéPredictors.

All of our code is in our GitHub repository, https://github.com/KevinShilla/PokePredictors.
This notebook clones the repository and runs our pipeline step by step to reproduce our selected submission.

In [ ]:
!pip install -q scikit-learn==1.5.2 2> /dev/null

In [ ]:
REPO_URL = "https://github.com/KevinShilla/PokePredictors.git"
REPO_DIR = "/kaggle/working/PokePredictors"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

In [ ]:
import glob, os, sys
import sklearn
import pandas as pd

print("scikit-learn", sklearn.__version__)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from pokepredictors.data import build_background, load_type_chart
from pokepredictors.features import build_features
from pokepredictors.mirrors import apply_known_results, known_test_results
from pokepredictors.model import cross_validate, fit_and_predict
from pokepredictors.skill import find_similar_teams

DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
train_path = os.path.join(DATA_DIR, "train.jsonl")
test_path = os.path.join(DATA_DIR, "test.jsonl")
chart_path = os.path.join(DATA_DIR, "type_chart_gen2.csv")

## Data

We read the battles one at a time and build lookup tables from both files: the types and base stats of every species,
the 60 most common species, and every known Player 1 team, which we use to recognise players and to guess Player 2's hidden team.

In [ ]:
chart = load_type_chart(chart_path)
background = build_background([train_path, test_path])
similar = find_similar_teams(background["teams_by_lead"])

print("Species:", len(background["species"]))
print("Different Player 1 teams:", len(background["team_count"]))

## Features

589 features per battle: the state at turn 30, momentum and moves, surviving species, type matchups and Player 2's hidden team.

In [ ]:
train_ids, X, y, teams = build_features(train_path, background, chart)
test_ids, X_test, _, test_teams = build_features(test_path, background, chart)
X_test = X_test[X.columns]

print("Train:", X.shape, " Test:", X_test.shape)
print("Player 2 team found for", round(X["p2_match"].mean() * 100, 1), "% of training battles")
X.head()

## Validation

Five fold stratified cross validation on the training battles. The player skill features are rebuilt inside every fold,
so no battle ever sees its own result.

In [ ]:
scores = cross_validate(X, y, teams, similar)
pd.DataFrame([scores])

## Submission: player skill ensemble

For each of three ways of measuring player skill (similar teams, all possible Player 2 teams, clutch skill) we train
a Logistic Regression and two HistGradientBoosting models on all the training battles, blend them with weights 0.3 and 0.7,
and average the three blends. Test battles that are exact copies of training battles take the known result.

Kaggle score: public 0.69000, private 0.69457.

Our submitted file was generated on a Windows laptop. Kaggle runs on Linux with a different processor, and tiny floating point
differences between the two machines can flip predictions that sit almost exactly at 50%, so a few dozen of the 5,000
predictions may differ from the submitted file. The method and the accuracy are the same.

In [ ]:
predictions = fit_and_predict(X, y, teams, X_test, test_teams, similar)
known = known_test_results(train_path, test_path)
apply_known_results(predictions, test_ids, known)

submission = pd.DataFrame({"battle_id": test_ids, "player_won": (predictions["final"] > 0.5).astype(int)})
submission.to_csv("/kaggle/working/submission.csv", index=False)

reference = pd.read_csv(os.path.join(REPO_DIR, "submissions.csv"))
matching = (reference["player_won"].values == submission["player_won"].values).sum()
print("Predictions matching our Kaggle submission:", matching, "of", len(submission), "(" + str(round(matching / len(submission) * 100, 1)) + "%)")
submission.head()